# LeetCode 3174 — Clear Digits bằng Python

Link đề: https://leetcode.com/problems/clear-digits/ · Độ khó: **Easy** · Chủ đề: Stack, String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def clearDigits(self, s: str) -> str:
        ...
```

Ghi chú: đề và ví dụ được chép/dịch theo đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code chạy thật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi lần **xoá chữ số đầu tiên** cùng với **ký tự không phải chữ số gần nhất bên trái** nó; lặp lại đến khi hết chữ số rồi trả về chuỗi còn lại.
2. Nắm các công cụ Python: `str.isdigit()`, **cắt chuỗi** `s[:i] + s[i+1:]` (chuỗi là bất biến), `list` dùng làm **stack** (`append`, `pop`, `[-1]`), `"".join(...)`, `enumerate`, và `re.sub` cho một cách làm phụ.
3. Đi từ cách **mô phỏng đúng như đề** (tìm chữ số đầu tiên rồi cắt chuỗi) → **stack một lượt** → **stack tại chỗ bằng một con trỏ ghi** → **duyệt từ phải sang trái với bộ đếm bỏ qua** → **regex lặp**.
4. Hiểu mấu chốt: "ký tự không phải chữ số **gần nhất** bên trái" chính là **phần tử trên cùng của stack** — đây là bài toán khớp ngoặc trá hình (chữ cái = `(`, chữ số = `)`).
5. Tránh các lỗi kinh điển: chỉ xoá chữ số mà quên xoá chữ cái bên trái, xoá chữ cái đầu chuỗi thay vì chữ cái **gần nhất**, vừa lặp vừa sửa danh sách, và nhận diện chữ số bằng cách viết thiếu `'0'`.

## 2. Đề bài & ví dụ

Cho một chuỗi `s`. Hãy xoá **tất cả** chữ số bằng cách lặp lại thao tác sau:

- Xoá chữ số **đầu tiên** (từ trái sang) **và** ký tự **không phải chữ số gần nhất** nằm **bên trái** nó.

Trả về chuỗi còn lại sau khi xoá hết chữ số.

Lưu ý: không thể thực hiện thao tác trên một chữ số nếu bên trái nó không có ký tự nào không phải chữ số (nhưng đề đảm bảo input luôn xoá được hết chữ số).

**Ví dụ 1**

- Input: `s = "abc"`
- Output: `"abc"`
- Giải thích: không có chữ số nào, chuỗi giữ nguyên.

**Ví dụ 2**

- Input: `s = "cb34"`
- Output: `""`
- Giải thích: thao tác với `s[2]` (chữ số `3`) xoá `3` và `b` → `"c4"`. Rồi thao tác với `s[1]` (chữ số `4`) xoá `4` và `c` → `""`.

**Ràng buộc:** `1 <= s.length <= 100`; `s` chỉ gồm chữ cái thường và chữ số; đảm bảo có thể xoá hết các chữ số.

In [1]:
import random
import re
import string

examples = [
    ("abc", "abc"),
    ("cb34", ""),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", repr(expected))

'abc' -> kỳ vọng 'abc'
'cb34' -> kỳ vọng ''


## 3. Kiến thức nền

### 3.1. Nhận biết chữ số: `str.isdigit()`

`ch.isdigit()` trả `True` nếu ký tự `ch` là chữ số (`'0'` đến `'9'`), ngược lại `False`. Với đề này, ký tự không phải chữ số chính là chữ cái thường.

In [2]:
for ch in "a1z0":
    print(repr(ch), ch.isdigit())       # a False, 1 True, z False, 0 True
print("5".isdigit(), "x".isdigit())     # True False

'a' False
'1' True
'z' False
'0' True
True False


### 3.2. Chuỗi là **bất biến**: cắt và ghép

Ta **không thể** sửa một ký tự của `str` tại chỗ (`s[0] = "x"` báo lỗi). Muốn "xoá" ký tự thứ `i`, ta tạo chuỗi **mới** bằng cách ghép phần trước `s[:i]` và phần sau `s[i+1:]`. Mỗi lần cắt-ghép tốn `O(n)` vì phải chép lại chuỗi.

In [3]:
s = "abc12"
i = 2
print(s[:i], "|", s[i + 1:])             # ab | 12
print(s[:i] + s[i + 1:])                 # ab12   (đã bỏ ký tự thứ 2)
try:
    s[0] = "x"
except TypeError as e:
    print("TypeError:", e)

ab | 12
ab12
TypeError: 'str' object does not support item assignment


### 3.3. `list` làm stack: `append`, `pop`, `[-1]`

**Stack** (ngăn xếp) là cấu trúc "vào sau, ra trước" (LIFO). Với `list` của Python:

- `st.append(x)` đẩy `x` lên **đỉnh** stack.
- `st.pop()` lấy và bỏ phần tử ở đỉnh (cuối list) — `O(1)`.
- `st[-1]` nhìn đỉnh mà không bỏ.

Phần tử **gần nhất bên trái** của vị trí hiện tại, trong số những thứ ta đã duyệt qua mà chưa bị xoá, chính là đỉnh stack.

In [4]:
st = []
st.append("a")
st.append("b")
print(st, "đỉnh =", st[-1])              # ['a', 'b'] đỉnh = b
print("pop ->", st.pop(), "còn", st)     # pop -> b còn ['a']
print(len(st), bool(st))                 # 1 True   (stack rỗng thì bool(st) là False)

['a', 'b'] đỉnh = b
pop -> b còn ['a']
1 True


### 3.4. Ghép list ký tự thành chuỗi: `"".join(...)`

Sau khi xử lý xong, ta có list ký tự `['a', 'd']`. `"".join(list)` nối chúng lại thành chuỗi `"ad"` (dấu `""` là chuỗi chèn giữa các phần tử — ở đây là không chèn gì).

In [5]:
chars = ["a", "d"]
print("".join(chars))                    # ad
print("-".join(chars))                   # a-d  (để thấy rõ vai trò của chuỗi chèn giữa)
print(repr("".join([])))                 # ''   (list rỗng -> chuỗi rỗng, đúng với ví dụ 2)

ad
a-d
''


### 3.5. `enumerate` và vòng lặp có chỉ số

`for i, ch in enumerate(s)` cho cả chỉ số `i` lẫn ký tự `ch`. Ta sẽ dùng nó để tìm **vị trí chữ số đầu tiên** và khi trace.

In [6]:
for i, ch in enumerate("ab1"):
    print(i, repr(ch), ch.isdigit())
# 0 'a' False / 1 'b' False / 2 '1' True

0 'a' False
1 'b' False
2 '1' True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `s = "ab1c23d"`. Làm đúng như đề:

1. Chữ số đầu tiên là `1` (vị trí 2); ký tự không phải chữ số gần nhất bên trái là `b` → xoá cả hai: `"ab1c23d"` → `"ac23d"`.
2. Chữ số đầu tiên bây giờ là `2`; bên trái gần nhất là `c` → `"a3d"`.
3. Chữ số đầu tiên là `3`; bên trái gần nhất là `a` → `"d"`.

Kết quả `"d"`.

Nhận xét quan trọng: mỗi chữ số "ăn" đúng **một chữ cái ngay trước nó** (kể cả khi chữ cái đó đứng trước một chữ số khác đã bị ăn rồi). Nếu ta đi từ trái sang phải và **nhớ các chữ cái chưa bị ăn**, thì khi gặp chữ số, chữ cái bị ăn chính là **chữ cái nhớ gần nhất** (vừa mới thêm vào). Đó là hành vi của stack.

In [7]:
s = "ab1c23d"
kept = []
for ch in s:
    if ch.isdigit():
        eaten = kept.pop()               # chữ cái gần nhất chưa bị ăn
        print(f"gặp chữ số {ch}: ăn chữ cái {eaten!r}, còn {kept}")
    else:
        kept.append(ch)
        print(f"gặp chữ cái {ch!r}: nhớ lại, còn {kept}")
print("kết quả:", "".join(kept))

gặp chữ cái 'a': nhớ lại, còn ['a']
gặp chữ cái 'b': nhớ lại, còn ['a', 'b']
gặp chữ số 1: ăn chữ cái 'b', còn ['a']
gặp chữ cái 'c': nhớ lại, còn ['a', 'c']
gặp chữ số 2: ăn chữ cái 'c', còn ['a']
gặp chữ số 3: ăn chữ cái 'a', còn []
gặp chữ cái 'd': nhớ lại, còn ['d']
kết quả: d


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng như đề

**Ý tưởng.** Làm đúng từng chữ của đề: lặp đi lặp lại

1. Tìm vị trí `i` của chữ số **đầu tiên** (quét từ trái).
2. Nếu không còn chữ số → dừng.
3. Ký tự không phải chữ số gần nhất bên trái là `s[i-1]` (vì chữ số đầu tiên nên mọi ký tự bên trái nó đều là chữ cái). Xoá `s[i-1]` và `s[i]` bằng cách cắt chuỗi: `s = s[:i-1] + s[i+1:]`.

In [8]:
def clear_digits_simulate(s: str) -> str:
    while True:
        first = -1
        for i, ch in enumerate(s):
            if ch.isdigit():
                first = i                # vị trí chữ số đầu tiên
                break
        if first == -1:                  # hết chữ số
            return s
        s = s[:first - 1] + s[first + 1:]    # bỏ s[first-1] (chữ cái) và s[first] (chữ số)


for s, expected in examples:
    print(repr(clear_digits_simulate(s)), "(kỳ vọng", repr(expected), ")")
print(repr(clear_digits_simulate("ab1c23d")))        # 'd'

'abc' (kỳ vọng 'abc' )
'' (kỳ vọng '' )
'd'


**Độ phức tạp (Cách 1).** Gọi `n = len(s)`. Mỗi vòng `while` quét tối đa `n` ký tự để tìm chữ số đầu tiên và cắt-ghép tốn `O(n)`; có tối đa `n/2` vòng (mỗi vòng xoá một chữ số). ⇒ **Time `O(n²)`**, **Space `O(n)`** (chuỗi mới mỗi vòng).

**Nút thắt:** mỗi vòng ta **quét lại từ đầu** để tìm chữ số đầu tiên, dù phần trước đó đã biết toàn chữ cái. Ta có thể quét **một lần** và nhớ các chữ cái đã gặp.

## 6. Ý tưởng thuật toán / bước đột phá

Duyệt `s` **một lần từ trái sang phải**, giữ một **stack** `st` các chữ cái chưa bị xoá:

- gặp **chữ cái** → `append` lên đỉnh;
- gặp **chữ số** → `pop` đỉnh (chính là chữ cái gần nhất bên trái chưa bị xoá), còn chữ số thì **không** đưa vào stack (nó đã bị xoá).

Cuối cùng stack chứa đúng các ký tự còn lại **theo thứ tự ban đầu**. Mỗi ký tự được `append` tối đa một lần và `pop` tối đa một lần ⇒ `O(n)`.

Vì sao không cần "chữ số đầu tiên"? Vì ta xử lý chữ số **theo đúng thứ tự trái → phải**, nên chữ số đang xét luôn là chữ số đầu tiên **còn lại** tại thời điểm đó.

In [9]:
s = "cb34"
st = []
for ch in s:
    if ch.isdigit():
        st.pop()
    else:
        st.append(ch)
    print(f"sau {ch!r}: {st}")
print(repr("".join(st)))                 # ''

sau 'c': ['c']
sau 'b': ['c', 'b']
sau '3': ['c']
sau '4': []
''


## 7. Các cách tiếp theo

### Cách 2 — Stack bằng `list`

**Giải thích từng đoạn.**

1. `st = []`: stack rỗng.
2. `for ch in s`: xử lý từng ký tự theo thứ tự.
3. `if ch.isdigit(): st.pop()`: chữ số xoá chữ cái ở đỉnh.
4. `else: st.append(ch)`: chữ cái được giữ tạm.
5. `return "".join(st)`: ghép các chữ cái còn lại.

In [10]:
def clear_digits_stack(s: str) -> str:
    st = []
    for ch in s:
        if ch.isdigit():
            st.pop()                     # xoá chữ cái gần nhất bên trái
        else:
            st.append(ch)
    return "".join(st)


for s, expected in examples:
    print(repr(clear_digits_stack(s)), "(kỳ vọng", repr(expected), ")")

'abc' (kỳ vọng 'abc' )
'' (kỳ vọng '' )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionStack:
    def clearDigits(self, s: str) -> str:
        st = []
        for ch in s:
            if ch.isdigit():
                st.pop()
            else:
                st.append(ch)
        return "".join(st)


print(repr(SolutionStack().clearDigits("ab1c23d")))        # 'd'

'd'


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(n)`** (stack tối đa `n` ký tự).

### Cách 3 — Stack tại chỗ bằng con trỏ ghi `k`

**Giải thích.** Thay vì tạo list mới ta dùng chính mảng ký tự `arr = list(s)` và một số nguyên `k` là **kích thước stack**: `arr[0:k]` là nội dung stack. Chữ cái: `arr[k] = ch; k += 1`. Chữ số: `k -= 1` (bỏ đỉnh). Cuối cùng kết quả là `arr[:k]`. Đây là dạng "hai con trỏ" quen thuộc (đọc `i`, ghi `k`), tiết kiệm bộ nhớ phụ trong ngôn ngữ cho sửa mảng.

In [12]:
def clear_digits_inplace(s: str) -> str:
    arr = list(s)                        # mảng ký tự có thể sửa
    k = 0                                # arr[:k] là nội dung stack
    for ch in arr:
        if ch.isdigit():
            k -= 1                       # bỏ phần tử ở đỉnh (chỉ cần lùi con trỏ)
        else:
            arr[k] = ch                  # ghi đè lên chỗ trống (k <= vị trí đang đọc)
            k += 1
    return "".join(arr[:k])


for s, expected in examples:
    print(repr(clear_digits_inplace(s)), "(kỳ vọng", repr(expected), ")")

'abc' (kỳ vọng 'abc' )
'' (kỳ vọng '' )


Lưu ý: vì `k` không bao giờ vượt vị trí đang đọc nên ghi `arr[k]` không phá dữ liệu chưa đọc. (Ở đây ta lặp qua `arr` trong khi ghi vào `arr`; điều này an toàn vì chỉ ghi vào vị trí **đã đọc xong hoặc đang đọc**, không đổi độ dài list.)

**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** cho `list(s)` trong Python (trong ngôn ngữ mảng có thể sửa tại chỗ thì `O(1)` phụ).

### Cách 4 — Duyệt từ **phải sang trái** với bộ đếm bỏ qua

**Giải thích.** Đi ngược từ cuối chuỗi, giữ `skip` = số chữ số "đang chờ" xoá một chữ cái bên trái:

- gặp **chữ số** → `skip += 1`;
- gặp **chữ cái** và `skip > 0` → chữ cái này bị xoá: `skip -= 1`;
- gặp **chữ cái** và `skip == 0` → giữ lại.

Kết quả được thu thập theo thứ tự ngược nên cuối cùng phải đảo lại. Cách này không cần stack, chỉ cần một biến đếm — giống kỹ thuật bài ngoặc.

In [13]:
def clear_digits_right_to_left(s: str) -> str:
    skip = 0
    kept = []
    for ch in reversed(s):
        if ch.isdigit():
            skip += 1                    # có thêm một chữ số cần "ăn" một chữ cái bên trái
        elif skip > 0:
            skip -= 1                    # chữ cái này bị ăn
        else:
            kept.append(ch)              # chữ cái sống sót
    kept.reverse()                       # đã thu theo thứ tự ngược
    return "".join(kept)


for s, expected in examples:
    print(repr(clear_digits_right_to_left(s)), "(kỳ vọng", repr(expected), ")")
print(repr(clear_digits_right_to_left("ab1c23d")))     # 'd'

'abc' (kỳ vọng 'abc' )
'' (kỳ vọng '' )
'd'


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (list kết quả).

### Cách 5 — Regex lặp: xoá cặp "chữ cái + chữ số" liền nhau

**Giải thích.** Một chữ số **ngay sau** một chữ cái chắc chắn xoá chữ cái đó. Mẫu `[a-z]\d` khớp đúng cặp liền kề này. `re.sub(r"[a-z]\d", "", s)` xoá tất cả cặp như vậy; sau khi xoá, có thể xuất hiện cặp liền kề mới (ví dụ `"ab12"`: xoá `b1` còn `"a2"`), nên ta **lặp lại đến khi chuỗi không đổi**. Đây là cách ngắn nhưng chậm hơn trong trường hợp xấu.

Lưu ý: `re.sub` thay các khớp **không chồng lấn** từ trái sang phải, nên mỗi lượt có thể chưa xoá hết; vòng lặp `while` lo việc đó. Với input hợp lệ (mọi chữ số đều có chữ cái bên trái), việc xoá các cặp liền kề theo thứ tự nào cũng cho cùng kết quả như stack (giống xoá cặp `()` liền kề trong dãy ngoặc cân bằng). Test ngẫu nhiên ở mục 12 sẽ kiểm chứng.

In [14]:
def clear_digits_regex(s: str) -> str:
    pattern = re.compile(r"[a-z]\d")     # một chữ cái ngay trước một chữ số
    while True:
        t = pattern.sub("", s)           # xoá mọi cặp liền kề (không chồng lấn)
        if t == s:                       # không còn cặp nào -> xong
            return t
        s = t


for s, expected in examples:
    print(repr(clear_digits_regex(s)), "(kỳ vọng", repr(expected), ")")
print(repr(clear_digits_regex("ab1c23d")))             # 'd'

'abc' (kỳ vọng 'abc' )
'' (kỳ vọng '' )
'd'


**Độ phức tạp Cách 5:** mỗi lượt `sub` là `O(n)`; số lượt tối đa `O(n)` (ví dụ `"abcd1234"` mỗi lượt chỉ xoá một cặp) ⇒ **Time `O(n²)`** trường hợp xấu, **Space `O(n)`**. Với `n <= 100` vẫn rất nhanh.

## 8. TRACE từng bước

Ta trace Cách 2 (stack) trên `s = "ab1c23d"` và Cách 4 (phải sang trái) trên `s = "cb34"`.

Bảng trace Cách 2 với `s = "ab1c23d"`:

| i | ký tự | hành động | stack sau bước |
|---|---|---|---|
| 0 | a | chữ cái → push | `['a']` |
| 1 | b | chữ cái → push | `['a', 'b']` |
| 2 | 1 | chữ số → pop `'b'` | `['a']` |
| 3 | c | chữ cái → push | `['a', 'c']` |
| 4 | 2 | chữ số → pop `'c'` | `['a']` |
| 5 | 3 | chữ số → pop `'a'` | `[]` |
| 6 | d | chữ cái → push | `['d']` |

Kết quả `"d"`.

In [15]:
def trace_stack(s: str) -> str:
    st = []
    print(f"s = {s!r}")
    for i, ch in enumerate(s):
        if ch.isdigit():
            removed = st.pop()
            print(f"  i={i} ch={ch!r}: chữ số -> pop {removed!r}, stack = {st}")
        else:
            st.append(ch)
            print(f"  i={i} ch={ch!r}: chữ cái -> push, stack = {st}")
    res = "".join(st)
    print("  kết quả:", repr(res))
    return res


assert trace_stack("ab1c23d") == "d"

s = 'ab1c23d'
  i=0 ch='a': chữ cái -> push, stack = ['a']
  i=1 ch='b': chữ cái -> push, stack = ['a', 'b']
  i=2 ch='1': chữ số -> pop 'b', stack = ['a']
  i=3 ch='c': chữ cái -> push, stack = ['a', 'c']
  i=4 ch='2': chữ số -> pop 'c', stack = ['a']
  i=5 ch='3': chữ số -> pop 'a', stack = []
  i=6 ch='d': chữ cái -> push, stack = ['d']
  kết quả: 'd'


In [16]:
def trace_right_to_left(s: str) -> str:
    skip = 0
    kept = []
    print(f"s = {s!r}")
    for i in range(len(s) - 1, -1, -1):
        ch = s[i]
        if ch.isdigit():
            skip += 1
            print(f"  i={i} ch={ch!r}: chữ số -> skip = {skip}")
        elif skip > 0:
            skip -= 1
            print(f"  i={i} ch={ch!r}: chữ cái bị ăn -> skip = {skip}")
        else:
            kept.append(ch)
            print(f"  i={i} ch={ch!r}: chữ cái giữ lại -> kept = {kept}")
    kept.reverse()
    res = "".join(kept)
    print("  kết quả:", repr(res))
    return res


assert trace_right_to_left("cb34") == ""
assert trace_right_to_left("a1bc2") == "b"

s = 'cb34'
  i=3 ch='4': chữ số -> skip = 1
  i=2 ch='3': chữ số -> skip = 2
  i=1 ch='b': chữ cái bị ăn -> skip = 1
  i=0 ch='c': chữ cái bị ăn -> skip = 0
  kết quả: ''
s = 'a1bc2'
  i=4 ch='2': chữ số -> skip = 1
  i=3 ch='c': chữ cái bị ăn -> skip = 0
  i=2 ch='b': chữ cái giữ lại -> kept = ['b']
  i=1 ch='1': chữ số -> skip = 1
  i=0 ch='a': chữ cái bị ăn -> skip = 0
  kết quả: 'b'


Đọc trace phải-sang-trái với `"a1bc2"`: `2` → `skip=1`; `c` bị ăn (`skip=0`); `b` giữ; `1` → `skip=1`; `a` bị ăn. Còn `b`. Khớp với stack: `a`, pop (do `1`) → rỗng, `b`, `c`, pop (do `2`) → `['b']`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (stack bằng `list`): ngắn, rõ, `O(n)`.

In [17]:
class Solution:
    def clearDigits(self, s: str) -> str:
        st = []                          # stack các chữ cái chưa bị xoá
        for ch in s:
            if ch.isdigit():
                st.pop()                 # chữ số xoá chữ cái gần nhất bên trái
            else:
                st.append(ch)
        return "".join(st)


sol = Solution()
for s, expected in examples:
    print(repr(s), "->", repr(sol.clearDigits(s)))

'abc' -> 'abc'
'cb34' -> ''


## 10. Vì sao thuật toán đúng

- **Chữ cái gần nhất bên trái = đỉnh stack.** Tại thời điểm xử lý chữ số, stack chứa các chữ cái đã duyệt mà chưa bị xoá, theo thứ tự xuất hiện; chữ cái **gần nhất** (xa nhất về bên phải) chính là chữ cái được đẩy vào **sau cùng** = đỉnh.
- **Thứ tự xử lý khớp đề.** Đề luôn xoá chữ số **đầu tiên** còn lại; duyệt trái → phải xử lý chữ số theo đúng thứ tự đó, và các ký tự bên trái chữ số đang xét đều là chữ cái chưa bị xoá hoặc đã bị xoá bởi chữ số trước nó — đúng như stack.
- **Không bao giờ `pop` stack rỗng:** đề đảm bảo mỗi chữ số xoá được, tức khi gặp nó luôn còn ít nhất một chữ cái bên trái.
- **Thứ tự còn lại được giữ:** các chữ cái sống sót nằm trong stack theo thứ tự đẩy vào, tức thứ tự ban đầu.
- **Cách 4 tương đương** vì nó là bài khớp ngoặc nhìn từ phải: mỗi chữ số "mượn" chữ cái chưa bị mượn gần nhất bên trái.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` (`n <= 100`).

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng cắt chuỗi | `O(n²)` | `O(n)` |
| 2. Stack bằng `list` | `O(n)` | `O(n)` |
| 3. Stack tại chỗ (con trỏ `k`) | `O(n)` | `O(n)` (`O(1)` phụ nếu mảng sửa được) |
| 4. Phải sang trái + `skip` | `O(n)` | `O(n)` |
| 5. Regex lặp | `O(n²)` | `O(n)` |

Với `n <= 100` mọi cách đều tức thì; nhưng thói quen đúng là dùng stack `O(n)`.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
sol = Solution()
impls = {
    "simulate": clear_digits_simulate,
    "stack": clear_digits_stack,
    "inplace": clear_digits_inplace,
    "right_to_left": clear_digits_right_to_left,
    "regex": clear_digits_regex,
    "leetcode": sol.clearDigits,
}


def check_all(s, expected, label):
    for key, fn in impls.items():
        actual = fn(s)
        assert actual == expected, (label, key, s, actual, expected)
    print("PASS", label)


for idx, (s, expected) in enumerate(examples, 1):
    check_all(s, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

In [19]:
check_all("a", "a", "một chữ cái")
check_all("a1", "", "một cặp")
check_all("z9", "", "cặp với chữ số 9")
check_all("a0", "", "chữ số 0 cũng là chữ số")
check_all("ab12", "", "hai chữ cái rồi hai chữ số")
check_all("a1b2", "", "xen kẽ")
check_all("a1b", "b", "chữ cái dư ở cuối")
check_all("ab1", "a", "chữ cái dư ở đầu")
check_all("abc123", "", "ba chữ cái rồi ba chữ số")
check_all("ab1c23d", "d", "ví dụ tự dựng ở mục 4")
check_all("abcde", "abcde", "không có chữ số")
check_all("a" * 50 + "1" * 50, "", "độ dài tối đa: 50 chữ cái + 50 chữ số")
check_all("a1" * 50, "", "100 ký tự xen kẽ")
check_all("a" * 99 + "1", "a" * 98, "một chữ số ở cuối")

PASS một chữ cái
PASS một cặp
PASS cặp với chữ số 9
PASS chữ số 0 cũng là chữ số
PASS hai chữ cái rồi hai chữ số
PASS xen kẽ
PASS chữ cái dư ở cuối
PASS chữ cái dư ở đầu
PASS ba chữ cái rồi ba chữ số
PASS ví dụ tự dựng ở mục 4
PASS không có chữ số
PASS độ dài tối đa: 50 chữ cái + 50 chữ số
PASS 100 ký tự xen kẽ
PASS một chữ số ở cuối


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi phổ biến: (a) **chỉ xoá chữ số**, quên xoá chữ cái; (b) xoá chữ cái **đầu tiên** của chuỗi thay vì **gần nhất**; (c) **ghép đoạn** `"".join(ch for ch in s if ch.isalpha())` cũng chính là (a).

In [20]:
def wrong_only_digits(s):
    return "".join(ch for ch in s if not ch.isdigit())        # SAI: quên xoá chữ cái bên trái


def wrong_remove_first_letter(s):
    arr = list(s)
    for ch in s:
        if ch.isdigit():
            arr.remove(ch)               # xoá chữ số (cũng sai chỗ nếu trùng) ...
            arr.pop(0)                   # SAI: xoá chữ cái ĐẦU TIÊN, không phải gần nhất
    return "".join(arr)


print(wrong_only_digits("ab1c2"), "!=", Solution().clearDigits("ab1c2"))
print(wrong_remove_first_letter("ab1c2"), "!=", Solution().clearDigits("ab1c2"))
assert wrong_only_digits("ab1c2") != Solution().clearDigits("ab1c2")
assert wrong_remove_first_letter("ab1c2") != Solution().clearDigits("ab1c2")

abc != a
c != a


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Sinh chuỗi **hợp lệ**: duy trì `depth` = số chữ cái chưa bị xoá; chỉ được thêm chữ số khi `depth > 0`. Độ dài `1..100`, chỉ gồm chữ thường và chữ số. Với mỗi chuỗi, tính đáp án bằng một mô phỏng **hoàn toàn độc lập** (xoá đúng theo đề dạng cắt chuỗi) và so với tất cả cách.

In [21]:
def gen_valid(rng, n):
    out = []
    depth = 0
    p_digit = rng.choice([0.2, 0.5, 0.8])
    for _ in range(n):
        if depth > 0 and rng.random() < p_digit:
            out.append(rng.choice(string.digits))
            depth -= 1
        else:
            out.append(rng.choice(string.ascii_lowercase))
            depth += 1
    return "".join(out)


def literal_reference(s):
    # mô phỏng đúng chữ trong đề, tách bạch với mọi cách phía trên
    s = list(s)
    while any(c.isdigit() for c in s):
        i = next(j for j, c in enumerate(s) if c.isdigit())
        left = max(j for j in range(i) if not s[j].isdigit())
        for j in sorted([i, left], reverse=True):
            del s[j]
    return "".join(s)


rng = random.Random(3174)
total = 0
for trial in range(3000):
    n = rng.choice([1, 2, 3, 5, 10, 30, 99, 100])
    s = gen_valid(rng, n)
    assert 1 <= len(s) <= 100 and all(c in string.ascii_lowercase + string.digits for c in s)
    ref = literal_reference(s)
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s, fn(s), ref)
    total += 1
print("PASS", total, "chuỗi ngẫu nhiên hợp lệ")

PASS 3000 chuỗi ngẫu nhiên hợp lệ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ xoá chữ số, quên xoá chữ cái bên trái

In [22]:
def sai_chi_xoa_chu_so(s):
    res = ""
    for ch in s:
        if not ch.isdigit():
            res += ch                    # SAI: chữ cái bên trái chữ số không bị xoá
    return res


print("sai:", repr(sai_chi_xoa_chu_so("cb34")), "| đúng:", repr(Solution().clearDigits("cb34")))

sai: 'cb' | đúng: ''


**Vì sao sai:** mỗi thao tác xoá **hai** ký tự (một chữ số và một chữ cái). Chỉ lọc bỏ chữ số cho `"cb"` thay vì `""`. **Cách sửa:** khi gặp chữ số phải `pop` chữ cái ở đỉnh stack.

### Lỗi 2: Xoá chữ cái đầu chuỗi thay vì chữ cái gần nhất bên trái

In [23]:
def sai_xoa_chu_cai_dau(s):
    st = []
    for ch in s:
        if ch.isdigit():
            st.pop(0)                    # SAI: pop(0) lấy phần tử ĐẦU (xa nhất), không phải gần nhất
        else:
            st.append(ch)
    return "".join(st)


print("sai:", repr(sai_xoa_chu_cai_dau("ab1")), "| đúng:", repr(Solution().clearDigits("ab1")))

sai: 'b' | đúng: 'a'


**Vì sao sai:** `"ab1"` phải xoá `b` (gần `1` nhất) và còn `"a"`, nhưng `pop(0)` xoá `a` và còn `"b"`. `pop()` không đối số mới lấy **cuối** list. (Ngoài ra `pop(0)` còn tốn `O(n)`.) **Cách sửa:** dùng `st.pop()`.

### Lỗi 3: Vừa duyệt vừa xoá phần tử trong cùng danh sách

In [24]:
def sai_sua_khi_lap(s):
    arr = list(s)
    for i, ch in enumerate(arr):
        if ch.isdigit():
            del arr[i]                   # SAI: đổi độ dài list khi đang enumerate nó
            del arr[i - 1]               # làm các chỉ số sau bị lệch, phần tử kế tiếp bị bỏ qua
    return "".join(arr)


print("sai:", repr(sai_sua_khi_lap("ab12")), "| đúng:", repr(Solution().clearDigits("ab12")))

sai: 'a2' | đúng: ''


**Vì sao sai:** khi xoá hai phần tử, các phần tử sau dồn lên hai ô nhưng `enumerate` vẫn tăng `i` như cũ ⇒ **bỏ sót** ký tự. Với `"ab12"`: sau khi xử lý `1` còn `['a','2']`, vòng lặp nhảy qua `2`, trả về `"a2"` sai. **Cách sửa:** không sửa list đang lặp; dùng stack (một list khác) hoặc con trỏ ghi `k`.

### Lỗi 4: Nhận diện chữ số thiếu `'0'`

In [25]:
def sai_thieu_so_0(s):
    st = []
    for ch in s:
        if ch in "123456789":            # SAI: quên '0'
            st.pop()
        else:
            st.append(ch)
    return "".join(st)


print("sai:", repr(sai_thieu_so_0("a0")), "| đúng:", repr(Solution().clearDigits("a0")))

sai: 'a0' | đúng: ''


**Vì sao sai:** `'0'` cũng là chữ số nên cũng phải xoá một chữ cái. Bản sai đẩy `'0'` vào stack như chữ cái, trả `"a0"`. **Cách sửa:** dùng `ch.isdigit()` thay vì tự liệt kê.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xoá ký tự **gần nhất bên trái** khi gặp một ký hiệu đặc biệt" ⇒ **stack** (đỉnh = gần nhất). Cùng họ: bài ngoặc `()`, backspace `#`, xoá cặp trùng liền kề.
- Muốn tránh vừa lặp vừa sửa: duyệt `s` gốc, **xây** kết quả trong một list riêng.
- Muốn tiết kiệm bộ nhớ: dùng một con trỏ ghi `k` làm "đỉnh stack" (stack tại chỗ).
- Không cần tách "chữ số đầu tiên": xử lý trái → phải đã bảo đảm thứ tự.
- Nhìn "chữ cái = mở ngoặc, chữ số = đóng ngoặc" để nhớ rằng input hợp lệ chính là dãy ngoặc **cân bằng ở mọi tiền tố** (không có `)` thừa).
- `"".join(list)` luôn nhanh hơn cộng chuỗi `+=` lặp lại trong vòng lặp lớn.

## 15. Biến thể của bài

1. **Backspace String Compare (844):** `#` xoá ký tự liền trước; so sánh hai chuỗi sau khi xử lý.
2. **Remove All Adjacent Duplicates In String (1047)**, **Make The String Great (1544):** stack xoá cặp liền kề.
3. **Valid Parentheses (20):** stack khớp ngoặc.
4. Biến thể tự nghĩ: trả về **danh sách các cặp bị xoá** (chữ cái, chữ số) hoặc **số ký tự bị xoá**.
5. Nếu input có thể **không hợp lệ** (chữ số thừa): bỏ qua chữ số đó thay vì lỗi (`if st: st.pop()`).

In [26]:
def backspace_compare(s, t):
    def build(x):
        st = []
        for ch in x:
            if ch == "#":
                if st:
                    st.pop()
            else:
                st.append(ch)
        return st
    return build(s) == build(t)


def clear_digits_safe(s):
    # chữ số thừa (không có chữ cái bên trái) thì bỏ qua
    st = []
    for ch in s:
        if ch.isdigit():
            if st:
                st.pop()
        else:
            st.append(ch)
    return "".join(st)


def removed_pairs(s):
    st, pairs = [], []
    for ch in s:
        if ch.isdigit():
            pairs.append((st.pop(), ch))
        else:
            st.append(ch)
    return pairs


print(backspace_compare("ab#c", "ad#c"))            # True  (cả hai thành "ac")
print(backspace_compare("a#c", "b"))                # False
print(repr(clear_digits_safe("1a2b3")))             # ''
print(removed_pairs("ab1c23d"))                     # [('b', '1'), ('c', '2'), ('a', '3')]
assert backspace_compare("ab#c", "ad#c") is True
assert backspace_compare("a#c", "b") is False
assert clear_digits_safe("1a2b3") == ""
assert removed_pairs("ab1c23d") == [("b", "1"), ("c", "2"), ("a", "3")]

True
False
''
[('b', '1'), ('c', '2'), ('a', '3')]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng cắt chuỗi | làm đúng từng chữ đề | `O(n²)` | `O(n)` | để hiểu đề, đối chiếu |
| 2. Stack `list` | chữ cái push, chữ số pop | `O(n)` | `O(n)` | **nên nộp** |
| 3. Stack tại chỗ | con trỏ ghi `k` | `O(n)` | `O(n)` (phụ `O(1)`) | ngôn ngữ cho sửa mảng, tiết kiệm |
| 4. Phải → trái + `skip` | đếm số chữ số chờ xoá | `O(n)` | `O(n)` | khi muốn tránh stack |
| 5. Regex lặp | xoá cặp liền kề đến khi ổn định | `O(n²)` | `O(n)` | code ngắn, `n` nhỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"a1".isdigit()`, `"7".isdigit()`, `"abc"[:1] + "abc"[2:]`, `Solution().clearDigits("ab1")`, `Solution().clearDigits("a1b2c3d")`.

**Bài 2 (tính tay).** Với `s = "ab12c3d4e"`: ghi lại stack sau mỗi ký tự rồi đưa ra kết quả cuối.

**Bài 3 (điền TODO).** Hoàn thành `clear_digits_todo(s)` theo Cách 2.

In [27]:
def clear_digits_todo(s):
    # TODO 1: tạo stack rỗng st = []
    # TODO 2: duyệt từng ch trong s: nếu ch.isdigit() thì st.pop(), ngược lại st.append(ch)
    # TODO 3: trả về "".join(st)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `clear_digits_skip(s)` theo Cách 4 (duyệt phải → trái với biến `skip`) mà **không** dùng `reversed` (dùng `range(len(s) - 1, -1, -1)`), và trả về chuỗi đúng thứ tự.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_removed(s)` trả về **số ký tự bị xoá** (cả chữ số lẫn chữ cái). (b) Viết `clear_digits_k(s)` theo luật mới: mỗi chữ số `d` xoá **chính nó và `d` chữ cái gần nhất bên trái** (ví dụ `"abc2"` → `"a"`; nếu không đủ chữ cái thì xoá hết những gì có).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `False`, `True`, `"ac"`, `"a"`, `"d"` (với `"a1b2c3d"`: `a1`, `b2`, `c3` bị xoá hết, chỉ còn `d`). Bài 3: stack là một `list` rỗng; chữ số thì `pop()`, chữ cái thì `append`.
- **Mức 2 (vừa):** Bài 2: `a`, `ab`, `a` (do `1`), `[]` (do `2`), `c`, `[]` (do `3`), `d`, `[]` (do `4`), `e` ⇒ `"e"`. Bài 4: chạy `i` từ `len(s) - 1` xuống `0`, thu chữ cái sống sót rồi đảo list.
- **Mức 3 (sâu):** Bài 5(a): `len(s) - len(kết_quả)`. Bài 5(b): khi gặp chữ số `d = int(ch)`, lặp `d` lần `if st: st.pop()`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert "a1".isdigit() is False
assert "7".isdigit() is True
assert "abc"[:1] + "abc"[2:] == "ac"
assert Solution().clearDigits("ab1") == "a"
assert Solution().clearDigits("a1b2c3d") == "d"

# Bài 2
assert Solution().clearDigits("ab12c3d4e") == "e"


# Bài 3
def clear_digits_todo_ans(s):
    st = []
    for ch in s:
        if ch.isdigit():
            st.pop()
        else:
            st.append(ch)
    return "".join(st)


for s, expected in examples:
    assert clear_digits_todo_ans(s) == expected


# Bài 4
def clear_digits_skip(s):
    skip = 0
    kept = []
    for i in range(len(s) - 1, -1, -1):
        ch = s[i]
        if ch.isdigit():
            skip += 1
        elif skip > 0:
            skip -= 1
        else:
            kept.append(ch)
    return "".join(kept[::-1])


for s, expected in examples:
    assert clear_digits_skip(s) == expected
assert clear_digits_skip("ab12c3d4e") == "e"


# Bài 5(a)
def count_removed(s):
    return len(s) - len(Solution().clearDigits(s))


assert count_removed("cb34") == 4
assert count_removed("abc") == 0
assert count_removed("ab1c23d") == 6


# Bài 5(b)
def clear_digits_k(s):
    st = []
    for ch in s:
        if ch.isdigit():
            for _ in range(int(ch)):
                if st:
                    st.pop()
        else:
            st.append(ch)
    return "".join(st)


assert clear_digits_k("abc2") == "a"
assert clear_digits_k("ab5") == ""
assert clear_digits_k("a1b1c") == "c"
assert clear_digits_k("abc") == "abc"
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (stack: chữ cái đẩy vào, chữ số lấy ra)

```python
class Solution:
    def clearDigits(self, s: str) -> str:
        st = []
        for ch in s:
            if ch.isdigit():
                st.pop()
            else:
                st.append(ch)
        return "".join(st)
```

**Checklist**

- [ ] Mỗi chữ số xoá **hai** ký tự (nó và một chữ cái), không chỉ một?
- [ ] Xoá chữ cái **gần nhất** (đỉnh stack, `pop()` không đối số), không phải chữ cái đầu (`pop(0)`)?
- [ ] Không sửa list đang được lặp qua?
- [ ] Dùng `isdigit()` (có cả `'0'`) thay vì tự liệt kê chữ số?
- [ ] Đã thử `"abc"` (`"abc"`), `"cb34"` (`""`), `"ab1c23d"` (`"d"`) và chuỗi toàn cặp `"a1"*50`?
- [ ] Skeleton đúng `class Solution` / `def clearDigits(self, s: str) -> str`?